# Comparison of deconvolver calibrators

In this notebook, we compare the performance of the our linear calibrator
versus temperature scaling, vector scaling and dirichlet calibration.

## Imports and utility functions

In [1]:
import os
from pathlib import Path
import json
from functools import reduce

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
from sklearn.model_selection import KFold, ParameterGrid

from EDA.edautils import plot_deconvolution_results
from methyldl.deconvolution.evaluation import compute_deconvolution_metrics
from methyldl.deconvolution.linear_calibrator import LinearCalibrator
from EDA.dirichlet_calibration import DirichletCalibrator

%load_ext autoreload
%autoreload 2

In [2]:
def prod_len_dict_keys(dict_: dict) -> int:
    return reduce(lambda x, y: x * y, list(map(len, dict_.values())), 1)

In [3]:
def plot_mixtures_pred_vs_true(
    ground_truth_mixture: np.ndarray,
    predicted_mixtures: list[np.ndarray],
    predicted_mixture_labels: list[str],
    width: float = 0.2,
    title: str = "Predicted Mixtures vs Ground Truth Mixture",
    ctype_names: list[str] = None,
):
    """
    Plot the predicted mixtures against the ground truth mixture for a single sample.
    The function creates a bar plot where the x axis represents the cell types and the y axis represents the mixture proportions.
    Each predicted mixture is plotted as a separate bar. The ground truth mixture is plotted as red dots for each cell type.

    Args:
        ground_truth_mixture: A 1D array of shape (n_cell_types,) representing the true mixture proportions.
        predicted_mixtures: A list of 1D arrays, each of shape (n_cell_types,), representing the predicted mixture proportions from different models.
        predicted_mixture_labels: A list of strings representing the labels for each predicted mixture (e.g., model names).
    """
    assert len(predicted_mixtures) == len(
        predicted_mixture_labels
    ), "Number of predicted mixtures must match number of labels"
    assert all(
        pred.shape == ground_truth_mixture.shape for pred in predicted_mixtures
    ), "All predicted mixtures must have the same shape as the ground truth mixture"
    plt.figure(figsize=(8, 6))
    n_cell_types = len(ground_truth_mixture)
    n_predicted_mixtures = len(predicted_mixtures)
    x = np.arange(n_cell_types)  # start of the the label locations
    x_center = (
        x + width * (n_predicted_mixtures - 1) / 2
    )  # middle of the group of bars for each cell type

    # Plot the ground truth mixture as red dots
    plt.scatter(
        x_center, ground_truth_mixture, color="red", zorder=-1, label="Ground Truth"
    )

    # Plot each predicted mixture as a bar
    for i, (predicted_mixture, label) in enumerate(
        zip(predicted_mixtures, predicted_mixture_labels)
    ):
        plt.bar(x + i * width, predicted_mixture, width, label=label, alpha=0.7)

    # plot the cell types names on the x axis, rotated by 90 degrees
    if ctype_names is not None:
        plt.xticks(x_center, ctype_names, rotation=90)
    plt.ylabel("Mixture Proportions")
    plt.title(title)
    plt.legend()
    plt.grid()
    plt.tight_layout()
    plt.show()

In [4]:
def plot_heatmap(
    matrix: np.ndarray,
    title: str = "Heatmap",
    color_bar_label: str = "Probability",
    xlabel="Predicted Class",
    ylabel="True Class",
    x_ticks: list[str] = None,
    y_ticks: list[str] = None,
    vmin: float | None = None,
    vmax: float | None = None,
):
    """Plot a heatmap of the given matrix with cell type names on the axes."""
    plt.figure(figsize=(10, 8))
    plt.imshow(matrix, cmap="viridis", aspect="auto", vmin=vmin, vmax=vmax)
    plt.colorbar(label=color_bar_label)
    plt.xticks(
        ticks=np.arange(matrix.shape[1]),
        labels=(
            x_ticks if x_ticks is not None else [str(i) for i in range(matrix.shape[1])]
        ),
        rotation=90,
    )
    plt.yticks(
        ticks=np.arange(matrix.shape[0]),
        labels=(
            y_ticks if y_ticks is not None else [str(i) for i in range(matrix.shape[0])]
        ),
    )
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.tight_layout()
    plt.show()

## Data loading

In [5]:
with open("../App/labels_dict.json", "r") as f:
    labels_to_ctype_names = json.load(f)
labels_to_ctype_names = {int(k): v for k, v in labels_to_ctype_names.items()}
ctype_names_to_labels = {v: k for k, v in labels_to_ctype_names.items()}
ctype_names_list = list(ctype_names_to_labels.keys())

In [6]:
target_proportions_file = Path(
    "../Data/training_data/mixture_predictions/uxm/target_proportions.npz"
)
target_prop = np.load(target_proportions_file)["arr_0"]

In [7]:
uxm_pred_folder = Path("../Data/training_data/mixture_predictions/uxm")
uxm_train_file = uxm_pred_folder / "uxm_results_train.npz"
uxm_val_file = uxm_pred_folder / "uxm_results_valid.npz"
uxm_test_file = uxm_pred_folder / "uxm_results_test.npz"

# uxm_train_data = np.load(uxm_train_file)["arr_0"]
uxm_val_data = np.load(uxm_val_file)["arr_0"]
uxm_test_data = np.load(uxm_test_file)["arr_0"]

In [8]:
def wrapper_comp_metrics(test_pred: dict, round_: int = 6) -> pd.DataFrame:
    results = {
        method_name: compute_deconvolution_metrics(
            pred=pred, target=target_prop, class_names=ctype_names_list
        )
        for method_name, pred in test_pred.items()
    }
    results = {
        k: {metric: value for metric, value in v.items() if "per_class" not in metric}
        for k, v in results.items()
    }
    results_df = pd.DataFrame(results).T
    float_cols = [
        "mae",
        "mse",
        "kl",
        "max_error",
        "cosine_sim",
        "loa_lower",
        "loa_upper",
        "loa_width",
        "worst_class_loa_lower",
        "worst_class_loa_upper",
        "worst_class_loa_width",
    ]
    for col in float_cols:
        results_df[col] = results_df[col].astype(float).round(round_)
    results_df.sort_values("mse", inplace=True)
    return results_df, results

## Compare calibrators

### Linear calibrators

In [ ]:
linear_cal = LinearCalibrator()
linear_cal.fit(uxm_val_data, uxm_test_data)
uxm_test_pred_lin_cal_clip01_norm, raw_uxm_test_pred_lin_cal = linear_cal.predict(
    uxm_test_data, norm_method="clip01-normalize"
)
uxm_test_pred_lin_cal_clip0_norm, _ = linear_cal.predict(
    uxm_test_data, norm_method="clip0-normalize"
)
uxm_test_pred_lin_cal_simplex_proj, _ = linear_cal.predict(
    uxm_test_data, norm_method="simplex-projection"
)

In [ ]:
results_df, results = wrapper_comp_metrics(
    {
        "UXM + no cal": uxm_test_data,
        "UXM + Lin cal (clip01-norm)": uxm_test_pred_lin_cal_clip01_norm,
        "UXM + Lin cal (clip0-norm)": uxm_test_pred_lin_cal_clip0_norm,
        "UXM + Lin cal (simplex-projection)": uxm_test_pred_lin_cal_simplex_proj,
    }
)
results_df

In [ ]:
# # we find the index of the sample with the highest MSE for the "UXM + no cal" method
# sample_idx = np.argmax(np.sum((uxm_test_data - target_prop) ** 2, axis=1))
# we find the index of the sample with the highest MAE for the "UXM + no cal" method
sample_idx = np.argmax(np.sum(np.abs(uxm_test_data - target_prop), axis=1))

In [ ]:
# plot_mixtures_pred_vs_true(
#     ground_truth_mixture=target_prop[sample_idx],
#     predicted_mixtures=[
#         uxm_test_data[sample_idx],
#         uxm_test_pred_lin_cal_clip01_norm[sample_idx],
#         uxm_test_pred_lin_cal_clip0_norm[sample_idx],
#         uxm_test_pred_lin_cal_simplex_proj[sample_idx],
#     ],
#     predicted_mixture_labels=[
#         "UXM + no cal",
#         "UXM + Lin cal (clip01-norm)",
#         "UXM + Lin cal (clip0-norm)",
#         "UXM + Lin cal (simplex-projection)",
#     ],
#     title="UXM Predictions with and without Linear Calibration",
#     ctype_names=ctype_names_list
# )

### Trained calibrators

In [9]:
# global settings
SCHEDULER = "plateau"
PLATEAU_FACTOR = 0.5
PLATEAU_PATIENCE = 10
LOG_TRANSFORM = True
BATCH_SIZE = None  # whole dataset
OPTIMIZER = "adam"
PATIENCE = 15
TOL = 1e-4
N_FOLDS = 3
DEVICE = "cuda"
CALIBRATOR_PARAMS = {
    "scheduler": SCHEDULER,
    "log_transform": LOG_TRANSFORM,
    "batch_size": BATCH_SIZE,
    "optimizer": OPTIMIZER,
    "patience": PATIENCE,
    "plateau_factor": PLATEAU_FACTOR,
    "plateau_patience": PLATEAU_PATIENCE,
    "tol": TOL,
    "verbose": False,
    "device": DEVICE,
}

#### Temperature scaling

In [ ]:
# Temperature scaling on the log prob grid search
METHOD = "temperature"
param_grid = ParameterGrid(
    {
        "reg_lambda": [0.0, 1e-4],
        "reg_mu": [None],
        "lr": [1e-1, 1e-2, 1e-3, 1e-4],
        "max_iter": [500],
    }
)

results = []
n_loops = len(param_grid) * N_FOLDS
best_val_loss = np.inf
best_temp_scal_params = None
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
with tqdm(total=n_loops) as pbar:
    for params in param_grid:
        lr = params["lr"]
        max_iter = params["max_iter"]
        reg_lambda = params["reg_lambda"]
        reg_mu = params["reg_mu"]
        fold_val_losses = []
        fold_val_mses = []
        max_n_trained_epochs = 0
        for train_index, val_index in kf.split(uxm_val_data):
            X_tr_fold, X_val_fold = uxm_val_data[train_index], uxm_val_data[val_index]
            y_tr_fold, y_val_fold = target_prop[train_index], target_prop[val_index]
            temp_scal = DirichletCalibrator(
                method=METHOD,
                lr=lr,
                max_iter=max_iter,
                reg_lambda=reg_lambda,
                reg_mu=reg_mu,
                **CALIBRATOR_PARAMS
            )
            temp_scal.fit(
                X=X_tr_fold,
                y=y_tr_fold,
                X_val=X_val_fold,
                y_val=y_val_fold,
                report_every=10,
            )
            fold_val_losses.append(temp_scal.best_metrics_["val_loss"])
            fold_val_mses.append(temp_scal.best_metrics_["val_mse"])
            max_n_trained_epochs = max(
                max_n_trained_epochs, temp_scal.history_["epoch"][-1] + 1
            )
            pbar.update(1)
        avg_best_val_loss = np.mean(fold_val_losses)
        results.append(
            {
                # Hyperparameters
                "reg_lambda": reg_lambda,
                "reg_mu": reg_mu,
                "lr": lr,
                "max_iter": max_iter,
                # Key metrics
                "avg_best_val_loss": avg_best_val_loss,
                "avg_best_val_mse": np.mean(fold_val_mses),
                "max_n_epochs_trained": max_n_trained_epochs,
                # Fitted model (not in DataFrame columns, but accessible)
            }
        )
        if avg_best_val_loss < best_val_loss:
            best_val_loss = avg_best_val_loss
            best_temp_scal_params = params
results_df_temp_scal = pd.DataFrame(results)
results_df_temp_scal.sort_values("avg_best_val_loss", inplace=True)
results_df_temp_scal

  0%|          | 0/6 [00:00<?, ?it/s]

,reg_lambda,reg_mu,lr,max_iter,avg_best_val_loss,avg_best_val_mse,max_n_epochs_trained
1,0.0001,None,0.1,500,1.530778,0.000227,17
0,0.0000,None,0.1,500,1.530779,0.000227,17


#### Vector scaling on the log prop

In [ ]:
# Vector scaling on the log prob grid search
METHOD = "diagonal"
param_grid = ParameterGrid(
    {
        "reg_lambda": [0.0, 1e-3, 1e-2, 1e-1],
        "reg_mu": [None],
        "lr": [1e-2, 1e-4, 1e-3],
        "max_iter": [1000],
    }
)

results = []
n_loops = len(param_grid) * N_FOLDS
best_val_loss = float("inf")
best_vect_scal_params = None
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
with tqdm(total=n_loops) as pbar:
    for params in param_grid:
        lr = params["lr"]
        max_iter = params["max_iter"]
        reg_lambda = params["reg_lambda"]
        reg_mu = params["reg_mu"]
        fold_val_losses = []
        fold_val_mses = []
        max_n_trained_epochs = 0
        for train_index, val_index in kf.split(uxm_val_data):
            X_tr_fold, X_val_fold = uxm_val_data[train_index], uxm_val_data[val_index]
            y_tr_fold, y_val_fold = target_prop[train_index], target_prop[val_index]
            vect_log_dir_cal = DirichletCalibrator(
                method=METHOD,
                lr=lr,
                max_iter=max_iter,
                reg_lambda=reg_lambda,
                reg_mu=reg_mu,
                **CALIBRATOR_PARAMS
            )
            vect_log_dir_cal.fit(
                X=X_tr_fold,
                y=y_tr_fold,
                X_val=X_val_fold,
                y_val=y_val_fold,
                report_every=10,
            )
            fold_val_losses.append(vect_log_dir_cal.best_metrics_["val_loss"])
            fold_val_mses.append(vect_log_dir_cal.best_metrics_["val_mse"])
            max_n_trained_epochs = max(
                max_n_trained_epochs, vect_log_dir_cal.history_["epoch"][-1] + 1
            )
            pbar.update(1)
        avg_val_loss = np.mean(fold_val_losses)
        results.append(
            {
                # Hyperparameters
                "reg_lambda": reg_lambda,
                "reg_mu": reg_mu,
                "lr": lr,
                "max_iter": max_iter,
                # Key metrics
                "avg_best_val_loss": avg_val_loss,
                "avg_best_val_mse": np.mean(fold_val_mses),
                "max_n_epochs_trained": max_n_trained_epochs,
            }
        )
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            best_vect_scal_params = params

results_df_vect_log_dir_cal = pd.DataFrame(results)
results_df_vect_log_dir_cal.sort_values("avg_best_val_loss", inplace=True)
results_df_vect_log_dir_cal

  0%|          | 0/9 [00:00<?, ?it/s]

,reg_lambda,reg_mu,lr,max_iter,avg_best_val_loss,avg_best_val_mse,max_n_epochs_trained
2,0.0,None,0.0010,1000,1.527684,0.000192,223
0,0.0,None,0.0100,1000,1.533141,0.000191,38
1,0.0,None,0.0001,1000,1.534892,0.000220,1000


best is {'lr': 1e-03, 'max_iter': 500, 'reg_lambda': 1e-3, 'reg_mu': None}

#### MS on the log prob

In [ ]:
# Matrix scaling on the log prob (Dir cal) grid search
METHOD = "full"
param_grid = ParameterGrid(
    {
        "reg_lambda": [0, 100, 1000, 5000, 10000],
        "reg_mu": [None, 0, 1, 10],
        "lr": [1e-4, 1e-5, 1e-6, 1e-7],
        "max_iter": [500],
    }
)

results = []
n_loops = len(param_grid) * N_FOLDS
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
best_val_loss = float("inf")
best_dir_cal_params = None
with tqdm(total=n_loops) as pbar:
    for params in param_grid:
        lr = params["lr"]
        max_iter = params["max_iter"]
        reg_lambda = params["reg_lambda"]
        reg_mu = params["reg_mu"]
        fold_val_losses = []
        fold_val_mses = []
        max_n_trained_epochs = 0
        for train_index, val_index in kf.split(uxm_val_data):
            X_tr_fold, X_val_fold = uxm_val_data[train_index], uxm_val_data[val_index]
            y_tr_fold, y_val_fold = target_prop[train_index], target_prop[val_index]
            mat_log_dir_cal = DirichletCalibrator(
                method=METHOD,
                lr=lr,
                max_iter=max_iter,
                reg_lambda=reg_lambda,
                reg_mu=reg_mu,
                **CALIBRATOR_PARAMS
            )
            mat_log_dir_cal.fit(
                X=X_tr_fold,
                y=y_tr_fold,
                X_val=X_val_fold,
                y_val=y_val_fold,
                report_every=10,
            )
            fold_val_losses.append(mat_log_dir_cal.best_metrics_["val_loss"])
            fold_val_mses.append(mat_log_dir_cal.best_metrics_["val_mse"])
            max_n_trained_epochs = max(
                max_n_trained_epochs, mat_log_dir_cal.history_["epoch"][-1] + 1
            )
            pbar.update(1)
        avg_val_loss = np.mean(fold_val_losses)
        results.append(
            {
                # Hyperparameters
                "reg_lambda": reg_lambda,
                "reg_mu": reg_mu,
                "lr": lr,
                "max_iter": max_iter,
                # Key metrics
                "avg_best_val_loss": avg_val_loss,
                "avg_best_val_mse": np.mean(fold_val_mses),
                "max_n_epochs_trained": max_n_trained_epochs,
            }
        )
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            best_dir_cal_params = params
results_df_mat_log_dir_cal = pd.DataFrame(results)
results_df_mat_log_dir_cal.sort_values("avg_best_val_loss", inplace=True)
results_df_mat_log_dir_cal

  0%|          | 0/48 [00:00<?, ?it/s]

,reg_lambda,reg_mu,lr,max_iter,avg_best_val_loss,avg_best_val_mse,max_n_epochs_trained
9,10000,0.0,1.000000e-06,500,1.538274,0.000177,128
10,10000,1.0,1.000000e-06,500,1.538274,0.000177,128
11,10000,10.0,1.000000e-06,500,1.538274,0.000177,128
4,10000,NaN,1.000000e-05,500,1.538488,0.000187,27
8,10000,NaN,1.000000e-06,500,1.541251,0.000198,155
13,10000,0.0,1.000000e-07,500,1.541644,0.000200,500
14,10000,1.0,1.000000e-07,500,1.541644,0.000200,500
15,10000,10.0,1.000000e-07,500,1.541644,0.000200,500
7,10000,10.0,1.000000e-05,500,1.543108,0.000186,27
6,10000,1.0,1.000000e-05,500,1.543108,0.000186,27


### Compare metrics across calibrators

In [17]:
best_temp_scal_params = {"lr": 1e-02, "max_iter": 500, "reg_lambda": 0, "reg_mu": None}
best_vect_scal_params = {
    "lr": 1e-03,
    "max_iter": 500,
    "reg_lambda": 1e-3,
    "reg_mu": None,
}
best_mat_dir_cal_params = {
    "lr": 1e-05,
    "max_iter": 500,
    "reg_lambda": 5000,
    "reg_mu": None,
}

We train with the best hyperparameters for each calibrator.

In [18]:
# Temperature scaling
best_temp_scal = DirichletCalibrator(
    method="temperature", **best_temp_scal_params, **CALIBRATOR_PARAMS
)
best_temp_scal.fit(X=uxm_val_data, y=target_prop, report_every=10)

,method,'temperature'
,log_transform,True
,reg_lambda,0
,reg_mu,None
,optimizer,'adam'
,lr,0.01
,scheduler,'plateau'
,max_iter,500
,batch_size,None
,patience,15
,tol,0.0001


In [19]:
# Vector scaling
best_vect_scal = DirichletCalibrator(
    method="diagonal", **best_vect_scal_params, **CALIBRATOR_PARAMS
)
best_vect_scal.fit(X=uxm_val_data, y=target_prop, report_every=10)

,method,'diagonal'
,log_transform,True
,reg_lambda,0.001
,reg_mu,None
,optimizer,'adam'
,lr,0.001
,scheduler,'plateau'
,max_iter,500
,batch_size,None
,patience,15
,tol,0.0001


In [20]:
# Dirichlet calibration
best_mat_dir_cal = DirichletCalibrator(
    method="full", **best_mat_dir_cal_params, **CALIBRATOR_PARAMS
)
best_mat_dir_cal.fit(X=uxm_val_data, y=target_prop, report_every=10)

,method,'full'
,log_transform,True
,reg_lambda,5000
,reg_mu,None
,optimizer,'adam'
,lr,1e-05
,scheduler,'plateau'
,max_iter,500
,batch_size,None
,patience,15
,tol,0.0001


In [21]:
# linear calibrators
linear_cal = LinearCalibrator()
linear_cal.fit(uxm_val_data, target_prop)

In [22]:
# inferences with the best calibrators
uxm_test_pred_lin_cal_clip01_norm, raw_uxm_test_pred_lin_cal = linear_cal.predict(
    uxm_test_data, norm_method="clip01-normalize"
)
uxm_test_pred_lin_cal_clip0_norm, _ = linear_cal.predict(
    uxm_test_data, norm_method="clip0-normalize"
)
uxm_test_pred_lin_cal_simplex_proj, _ = linear_cal.predict(
    uxm_test_data, norm_method="simplex-projection"
)
uxm_test_pred_temp_scal = best_temp_scal.predict(uxm_test_data)
uxm_test_pred_vect_scal = best_vect_scal.predict(uxm_test_data)
uxm_test_pred_mat_dir_cal = best_mat_dir_cal.predict(uxm_test_data)

In [23]:
results_df, results_dict = wrapper_comp_metrics(
    {
        "No calibration": uxm_test_data,
        "Linear Cal. (clip01-norm)": uxm_test_pred_lin_cal_clip01_norm,
        "Linear Cal. (clip0-norm)": uxm_test_pred_lin_cal_clip0_norm,
        "Linear Cal. (simplex-projection)": uxm_test_pred_lin_cal_simplex_proj,
        "Temperature scaling": uxm_test_pred_temp_scal,
        "Vector scaling": uxm_test_pred_vect_scal,
        "Dirichlet calibration": uxm_test_pred_mat_dir_cal,
    }
)
results_df.sort_values("mse")

,mae,mse,kl,max_error,cosine_sim,overall_r2,loa_lower,loa_upper,loa_width,worst_class_idx,worst_class_name,worst_class_loa_lower,worst_class_loa_upper,worst_class_loa_width
Linear Cal. (simplex-projection),0.004211,0.000208,0.124543,0.363513,0.986401,0.974931,-0.028240,0.028240,0.056480,11,Colon-Fibro,-0.063390,0.085064,0.148454
Vector scaling,0.004867,0.000219,0.111390,0.361775,0.984500,0.973566,-0.028999,0.028999,0.057997,11,Colon-Fibro,-0.055376,0.069524,0.124900
Dirichlet calibration,0.005332,0.000243,0.135165,0.356040,0.987498,0.970682,-0.030540,0.030540,0.061079,11,Colon-Fibro,-0.060708,0.070525,0.131232
Linear Cal. (clip0-norm),0.005028,0.000254,0.133447,0.411706,0.985780,0.969282,-0.031260,0.031260,0.062520,11,Colon-Fibro,-0.063620,0.085536,0.149156
Linear Cal. (clip01-norm),0.005033,0.000255,0.133534,0.411706,0.985778,0.969242,-0.031280,0.031280,0.062561,11,Colon-Fibro,-0.063627,0.085552,0.149179
Temperature scaling,0.005132,0.000272,0.112483,0.389757,0.981257,0.967192,-0.032306,0.032306,0.064612,11,Colon-Fibro,-0.071708,0.101430,0.173138
No calibration,0.005563,0.000294,0.145682,0.513800,0.985040,0.964509,-0.033601,0.033601,0.067202,11,Colon-Fibro,-0.068714,0.095688,0.164402


In [ ]:
results_df.columns

In [24]:
# print the latex formatting for the results dataframe


def fmt_r2(val):
    return f"{val * 100:.2f}"


def fmt_loa(lower, upper):
    return f"[{lower:.4f}, {upper:.4f}]"


def fmt_val(val, decimals=6):
    return f"{val:.{decimals}f}"


metrics_config = {
    "r2": {
        "col": "overall_r2",
        "higher_better": True,
        "fmt": lambda row: fmt_r2(row["overall_r2"]),
    },
    "loa": {
        "col": "loa_width",
        "higher_better": False,
        "fmt": lambda row: fmt_loa(row["loa_lower"], row["loa_upper"]),
    },
    "loa_worst": {
        "col": "worst_class_loa_width",
        "higher_better": False,
        "fmt": lambda row: fmt_loa(
            row["worst_class_loa_lower"], row["worst_class_loa_upper"]
        ),
    },
    "mae": {
        "col": "mae",
        "higher_better": False,
        "fmt": lambda row: fmt_val(row["mae"]),
    },
    "mse": {
        "col": "mse",
        "higher_better": False,
        "fmt": lambda row: fmt_val(row["mse"]),
    },
    "kl": {"col": "kl", "higher_better": False, "fmt": lambda row: fmt_val(row["kl"])},
}

# Find best and second best for each metric
rankings = {}
for key, config in metrics_config.items():
    col = config["col"]
    ascending = not config["higher_better"]
    sorted_idx = (
        results_df[col].astype(float).sort_values(ascending=ascending).index.tolist()
    )
    rankings[key] = {"best": sorted_idx[0], "second": sorted_idx[1]}

# Generate LaTeX content lines
for method_name, row in results_df.iterrows():
    values = []
    for key, config in metrics_config.items():
        formatted = config["fmt"](row)
        if method_name == rankings[key]["best"]:
            formatted = f"\\textbf{{{formatted}}}"
        elif method_name == rankings[key]["second"]:
            formatted = f"\\underline{{{formatted}}}"
        values.append(formatted)
    line = f"      {method_name:<40s} & {' & '.join(values)} \\\\"
    print(line)

      Linear Cal. (simplex-projection)         & \textbf{97.49} & \textbf{[-0.0282, 0.0282]} & [-0.0634, 0.0851] & \textbf{0.004211} & \textbf{0.000208} & 0.124543 \\
      Vector scaling                           & \underline{97.36} & \underline{[-0.0290, 0.0290]} & \textbf{[-0.0554, 0.0695]} & \underline{0.004867} & \underline{0.000219} & \textbf{0.111390} \\
      Dirichlet calibration                    & 97.07 & [-0.0305, 0.0305] & \underline{[-0.0607, 0.0705]} & 0.005332 & 0.000243 & 0.135165 \\
      Linear Cal. (clip0-norm)                 & 96.93 & [-0.0313, 0.0313] & [-0.0636, 0.0855] & 0.005028 & 0.000254 & 0.133447 \\
      Linear Cal. (clip01-norm)                & 96.92 & [-0.0313, 0.0313] & [-0.0636, 0.0856] & 0.005033 & 0.000255 & 0.133534 \\
      Temperature scaling                      & 96.72 & [-0.0323, 0.0323] & [-0.0717, 0.1014] & 0.005132 & 0.000272 & \underline{0.112483} \\
      No calibration                           & 96.45 & [-0.0336, 0.0336] & [-0.0687, 0

### Vect scaling on the prob with sparsemax

This one is out of the scope of the paper

In [ ]:
vect_scal_nolog_sparsemax_cal = DirichletCalibrator(
    method="diagonal",
    lr=0.01,
    log_transform=False,
    max_iter=500,
    batch_size=100000,
    device="cuda",
    scheduler="cosine",
    optimizer="adam",
    reg_lambda=0.0,
    reg_mu=None,
    verbose=True,
    patience=50,
    tol=1e-7,
    normalization="sparsemax",
)
vect_scal_nolog_sparsemax_cal.fit(
    X=uxm_train_data,
    y=target_prop,
    X_val=uxm_val_data,
    y_val=target_prop,
    report_every=10,
)

In [ ]:
# Vector scaling on the log prob grid search
METHOD = "diagonal"
LOG_TRANSFORM = False
NORMALIZATION = "sparsemax"
param_grid = {
    "reg_lambda": [0.0],
    "reg_mu": [None],
    "lr": [0.01, 0.001],
    "max_iter": [500, 1000],
    "scheduler": ["cosine"],
    "batch_size": [100000],
    "patience": [50],
    "tol": [1e-7],
}

results = []
n_loops = prod_len_dict_keys(param_grid)
best_val_loss = np.inf
best_model_vect_nolog_sparsemax_dir_cal = None
with tqdm(total=n_loops) as pbar:
    for reg_lambda in param_grid["reg_lambda"]:
        for reg_mu in param_grid["reg_mu"]:
            for lr in param_grid["lr"]:
                for max_iter in param_grid["max_iter"]:
                    for scheduler in param_grid["scheduler"]:
                        for batch_size in param_grid["batch_size"]:
                            for patience in param_grid["patience"]:
                                for tol in param_grid["tol"]:
                                    vect_nolog_sparsemax_dir_cal = DirichletCalibrator(
                                        method=METHOD,
                                        lr=lr,
                                        log_transform=LOG_TRANSFORM,
                                        max_iter=max_iter,
                                        batch_size=batch_size,
                                        device="cuda",
                                        scheduler=scheduler,
                                        optimizer="adam",
                                        reg_lambda=reg_lambda,
                                        reg_mu=reg_mu,
                                        verbose=False,
                                        patience=patience,
                                        tol=tol,
                                        normalization=NORMALIZATION,
                                    )
                                    vect_nolog_sparsemax_dir_cal.fit(
                                        X=uxm_train_data,
                                        y=target_prop,
                                        X_val=uxm_val_data,
                                        y_val=target_prop,
                                        report_every=10,
                                    )
                                    results.append(
                                        {
                                            # Hyperparameters
                                            "reg_lambda": reg_lambda,
                                            "reg_mu": reg_mu,
                                            "lr": lr,
                                            "max_iter": max_iter,
                                            "scheduler": scheduler,
                                            "batch_size": batch_size,
                                            "patience": patience,
                                            "tol": tol,
                                            # Key metrics
                                            "best_train_loss": vect_nolog_sparsemax_dir_cal.best_metrics_[
                                                "train_loss"
                                            ],
                                            "best_val_loss": vect_nolog_sparsemax_dir_cal.best_metrics_[
                                                "val_loss"
                                            ],
                                            "best_train_mse": vect_nolog_sparsemax_dir_cal.best_metrics_[
                                                "train_mse"
                                            ],
                                            "best_val_mse": vect_nolog_sparsemax_dir_cal.best_metrics_[
                                                "val_mse"
                                            ],
                                            "best_epoch": vect_nolog_sparsemax_dir_cal.best_epoch_,
                                            "n_epochs_trained": vect_nolog_sparsemax_dir_cal.history_[
                                                "epoch"
                                            ][
                                                -1
                                            ]
                                            + 1,
                                            # Fitted model (not in DataFrame columns, but accessible)
                                        }
                                    )
                                    if (
                                        vect_nolog_sparsemax_dir_cal.best_metrics_[
                                            "val_loss"
                                        ]
                                        < best_val_loss
                                    ):
                                        best_val_loss = (
                                            vect_nolog_sparsemax_dir_cal.best_metrics_[
                                                "val_loss"
                                            ]
                                        )
                                        best_model_vect_nolog_sparsemax_dir_cal = (
                                            vect_nolog_sparsemax_dir_cal
                                        )
                                    pbar.update(1)
results_df = pd.DataFrame(results)
results_df.sort_values("best_val_mse")

In [ ]:
vect_scal_nolog_sparsemax_cal_test_pred = (
    best_model_vect_nolog_sparsemax_dir_cal.predict(uxm_test_data)
)
results_df, results = wrapper_comp_metrics(
    {
        "UXM + no cal": uxm_test_data,
        "UXM + affine cal (clip01-norm) (OG)": uxm_test_pred_lin_cal_clip01_norm,
        "UXM + affine cal (clip0-norm)": uxm_test_pred_lin_cal_clip0_norm,
        "UXM + affine cal (simplex-projection)": uxm_test_pred_lin_cal_simplex_proj,
        "UXM + Vect Scal no log prob + Sparsemax Cal": vect_scal_nolog_sparsemax_cal_test_pred,
        "UXM + Vect Scal log prob + Softmax Cal": vect_scal_log_softmax_cal_test_pred,
    }
)
results_df.sort_values("mse")

In [ ]:
results_df.to_csv(
    "../output/calibrators_comparison/global_comparison_1.csv", index=False
)